# Australia Demand Dashboard

Demand-focused view of the DCCEEW Petroleum Statistics database produced by `scripts/update_australia.py`. Covers monthly data from 2010-07 onward, all metrics, 42 distinct products.

## Sections

1. **Setup & data load** — imports, parquet load, helper slices
2. **Headline** — Australia's total petroleum demand over time
3. **Demand by product** — long-term trends for the major refined products
4. **Recent trends (last 24 months)** — zoomed-in view + MoM/YoY snapshot table
5. **Year-over-year growth** — what's accelerating, what's shrinking (trailing 12mo vs prior 12mo)
6. **Seasonality (index)** — monthly index per product, last 5 complete years collapsed (the seasonal *shape*)
6b. **Seasonality (by calendar year)** — one panel per product, one line per year, current year highlighted (the year-over-year *shift*)
7. **Apparent vs reported** — energy-balance data-quality sanity check
8. **Interactive product deep-dive** — dropdown picker over all products
9. **DCCEEW vs JODI** — cross-source comparison after unit alignment (kL)
10. **Unit conversion demo** — `analytics.units.convert` in action: ML / kL / kb / kbd / kt side-by-side
11. **DCCEEW vs Kayrros nowcaster (Australia jet)** — official sales vs flight-derived consumption, both in kbd

## Conventions

- All volumes are in **ML** (megalitres = million litres). DCCEEW does not publish a kt/bbl conversion in the source workbook.
- `Total` and `Other products` are synthetic aggregate rows in the source — they're filtered out of product-level charts so they don't dominate the scale.
- The `: total` / ` total` suffix on demand-side product names (e.g. `Diesel oil: total`) is a DCCEEW labelling quirk. Canonicalisation against a master product list is **Phase 4** work, deferred until India and JODI are also normalised.

## 1. Setup & data load

In [1]:
"""Imports + parquet load + the slices the rest of the notebook reuses."""
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def _resolve_project_root() -> Path:
    """Walk up from the current working directory until we find the project root.

    Robust to being launched from anywhere — JupyterLab from the repo root,
    VS Code from the notebooks/ folder, papermill from CI, etc.
    """
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_australia.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_australia.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "australia"
PARQUET_PATH = PROCESSED_DIR / "australia_petroleum_statistics.parquet"

df = pd.read_parquet(PARQUET_PATH)
df["date"] = pd.to_datetime(df["date"])

# Demand-only slice. This is the single most-used view in the notebook so we
# materialise it once and reuse, avoiding 8 repeats of the same filter.
demand = df[df["metric_type"] == "TOTDEMO"].copy()

# `Total` (= all products combined) and `Other products` (a residual bucket)
# are synthetic aggregate rows. They're useful for the headline chart but dwarf
# real products on small-multiple charts, so we keep a `demand_products` view
# without them for any per-product visual.
SYNTHETIC_PRODUCTS = {"Total", "Other products"}
demand_products = demand[~demand["product_native"].isin(SYNTHETIC_PRODUCTS)].copy()

print(f"Loaded: {len(df):,} rows  ({df['date'].min().date()}  ->  {df['date'].max().date()})")
print(f"Demand only: {len(demand):,} rows, {demand['product_native'].nunique()} distinct products")
print(f"Units in demand slice: {sorted(demand['unit'].unique())}")

Loaded: 15,498 rows  (2010-07-01  ->  2026-03-01)
Demand only: 3,402 rows, 18 distinct products
Units in demand slice: ['ML']


In [2]:
df

,date,country,country_name,source,metric_type,product_native,product,value,unit,source_file,updated_at,product_canonical,category
0,2010-07-01,AU,Australia,dceew_petroleum_statistics,CLOSTLV,Automotive gasoline,Automotive gasoline,1137.0,ML,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,None,None
1,2010-07-01,AU,Australia,dceew_petroleum_statistics,CLOSTLV,Aviation gasoline,Aviation gasoline,24.1,ML,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,Gasoline,Gasoline
2,2010-07-01,AU,Australia,dceew_petroleum_statistics,CLOSTLV,Aviation turbine fuel,Aviation turbine fuel,293.2,ML,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,None,None
3,2010-07-01,AU,Australia,dceew_petroleum_statistics,CLOSTLV,Crude oil and refinery feedstocks,Crude oil and refinery feedstocks,3191.5,ML,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,None,None
4,2010-07-01,AU,Australia,dceew_petroleum_statistics,CLOSTLV,Diesel oil,Diesel oil,938.6,ML,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15493,2026-03-01,AU,Australia,dceew_petroleum_statistics,X_STKCOVER,Fuel oil,Fuel oil,71.0,days,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,Fuel Oil,Fuel Oil
15494,2026-03-01,AU,Australia,dceew_petroleum_statistics,X_STKCOVER,LPG,LPG,74.0,days,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,None,None
15495,2026-03-01,AU,Australia,dceew_petroleum_statistics,X_STKCOVER,"Lubricating oils, greases & basestocks","Lubricating oils, greases & basestocks",52.0,days,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,None,None
15496,2026-03-01,AU,Australia,dceew_petroleum_statistics,X_STKCOVER,Other products,Other products,311.0,days,australian_petroleum_statistics_-_data_extract...,2026-05-18 14:26:44.618282,Others,Others


## 2. Headline — total Australian petroleum demand

All products combined, monthly, with a 12-month rolling average to strip out seasonality. Use this chart to answer: _is Australia using more or less oil overall?_

In [3]:
total = demand[demand["product_native"] == "Total"].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=total["date"], y=total["value"],
    mode="lines", name="Monthly demand",
))
# 12-month rolling mean is the analyst's standard de-seasonalising filter. It
# trades two weeks of recency for ~10x noise reduction.
fig.add_trace(go.Scatter(
    x=total["date"], y=total["value"].rolling(12, min_periods=1).mean(),
    mode="lines", name="12-month rolling avg", line=dict(dash="dash", width=2),
))
fig.update_layout(
    title="Australia — Total petroleum demand (TOTDEMO, all products)",
    template="plotly_white", height=440, hovermode="x unified",
    yaxis_title="ML / month",
)
fig.show()

# Quick verbal summary for the analyst
latest = total.iloc[-1]
year_ago = total[total["date"] == latest["date"] - pd.DateOffset(years=1)]
five_years_ago = total[total["date"] == latest["date"] - pd.DateOffset(years=5)]
print(f"Latest month: {latest['date'].strftime('%Y-%m')}  ->  {latest['value']:,.0f} ML")
if not year_ago.empty:
    yoy = (latest["value"] / year_ago.iloc[0]["value"] - 1) * 100
    print(f"  vs same month last year:    {yoy:+.2f}%  ({year_ago.iloc[0]['value']:,.0f} ML)")
if not five_years_ago.empty:
    fyo = (latest["value"] / five_years_ago.iloc[0]["value"] - 1) * 100
    print(f"  vs same month 5 years ago:  {fyo:+.2f}%  ({five_years_ago.iloc[0]['value']:,.0f} ML)")

Latest month: 2026-03  ->  5,422 ML
  vs same month last year:    +4.59%  (5,184 ML)
  vs same month 5 years ago:  +15.49%  (4,695 ML)


## 3. Demand by product (long-term)

Major refined products only. Use this chart to see which products are the largest, and their long-term trajectory.

In [4]:
# The six biggest demand categories at the top-level. We use the `: total` /
# ` total` labels because DCCEEW splits each into sub-products (e.g. Diesel
# oil: premium diesel) that we don't want to chart separately here.
MAJOR_PRODUCTS = [
    "Automotive gasoline total",
    "Diesel oil: total",
    "Aviation turbine fuel total",
    "LPG total",
    "Fuel oil",
    "Lubricating oils & greases",
]

mp = demand[demand["product_native"].isin(MAJOR_PRODUCTS)].sort_values(["product_native", "date"])

fig = px.line(
    mp, x="date", y="value", color="product_native",
    title="Australian demand by major product (TOTDEMO, ML/month)",
    labels={"value": "ML / month", "date": "", "product_native": "Product"},
)
fig.update_layout(template="plotly_white", height=480, hovermode="x unified")
fig.show()

## 4. Recent trends — last 24 months

Zoomed-in view + a snapshot table showing each product's latest demand, month-over-month %, and year-over-year %.

In [5]:
two_years_ago = demand["date"].max() - pd.DateOffset(months=23)
recent = demand[
    (demand["date"] >= two_years_ago)
    & (demand["product_native"].isin(MAJOR_PRODUCTS))
].sort_values(["product_native", "date"]).copy()

# Within each product, compute MoM (1 month) and YoY (12 months) percentage
# changes. groupby(...).pct_change() respects group boundaries, so we don't
# accidentally compute Aviation-fuel-jan-2025 / Diesel-feb-2026.
recent["mom_pct"] = recent.groupby("product_native")["value"].pct_change(periods=1) * 100
recent["yoy_pct"] = recent.groupby("product_native")["value"].pct_change(periods=12) * 100

fig = px.line(
    recent, x="date", y="value", color="product_native",
    title="Last 24 months — demand by major product",
    labels={"value": "ML / month", "date": "", "product_native": "Product"},
)
fig.update_layout(template="plotly_white", height=420, hovermode="x unified")
fig.show()

latest_month = recent["date"].max()
snap = (
    recent[recent["date"] == latest_month][["product_native", "value", "mom_pct", "yoy_pct"]]
    .sort_values("value", ascending=False)
    .reset_index(drop=True)
)
snap.columns = ["Product", "Latest demand (ML)", "MoM %", "YoY %"]
print(f"Snapshot — latest month: {latest_month.strftime('%Y-%m')}")
display(snap.style.format({
    "Latest demand (ML)": "{:,.0f}",
    "MoM %": "{:+.2f}",
    "YoY %": "{:+.2f}",
}))

Snapshot — latest month: 2026-03


,Product,Latest demand (ML),MoM %,YoY %
0,Diesel oil: total,"3,015",+15.25,+9.47
1,Automotive gasoline total,"1,360",+10.10,+0.32
2,Aviation turbine fuel total,739,-10.20,-8.88
3,LPG total,119,+24.84,+8.17
4,Fuel oil,87,+43.54,+32.57
5,Lubricating oils & greases,30,+13.26,+12.41


## 5. Year-over-year growth (all products)

Compares the trailing 12 months of demand to the 12 months before that. Aggregating to a full-year basis smooths out single-month volatility and gives a clean read on _direction_ per product. Excludes synthetic aggregate rows (`Total`, `Other products`).

In [6]:
end = demand["date"].max()
last12_start = end - pd.DateOffset(months=11)
prev12_end = last12_start - pd.DateOffset(months=1)
prev12_start = prev12_end - pd.DateOffset(months=11)

scope = demand_products  # synthetic-rows already excluded
last12 = scope[(scope["date"] >= last12_start) & (scope["date"] <= end)]
prev12 = scope[(scope["date"] >= prev12_start) & (scope["date"] <= prev12_end)]

yoy = pd.DataFrame({
    "last12": last12.groupby("product_native")["value"].sum(),
    "prev12": prev12.groupby("product_native")["value"].sum(),
})
# Only keep products with a meaningful prior baseline so the % isn't noise.
yoy = yoy[yoy["prev12"] > 100]
yoy["abs_change"] = yoy["last12"] - yoy["prev12"]
yoy["yoy_pct"] = (yoy["last12"] / yoy["prev12"] - 1) * 100
yoy = yoy.sort_values("yoy_pct")

fig = px.bar(
    yoy.reset_index(), x="yoy_pct", y="product_native", orientation="h",
    title=(
        f"YoY change in demand:  {last12_start.strftime('%Y-%m')} → {end.strftime('%Y-%m')}  "
        f"vs  {prev12_start.strftime('%Y-%m')} → {prev12_end.strftime('%Y-%m')}"
    ),
    labels={"yoy_pct": "YoY %", "product_native": ""},
    color="yoy_pct", color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
)
fig.update_layout(template="plotly_white", height=620, coloraxis_showscale=False)
fig.add_vline(x=0, line=dict(color="black", width=1))
fig.show()

display(yoy[["last12", "prev12", "abs_change", "yoy_pct"]].style.format({
    "last12": "{:,.0f}",
    "prev12": "{:,.0f}",
    "abs_change": "{:+,.0f}",
    "yoy_pct": "{:+.2f}%",
}))

,last12,prev12,abs_change,yoy_pct
product_native,,,,
Fuel oil,677,757,-80,-10.62%
Ethanol-blended fuel,"1,702","1,849",-147,-7.95%
Regular (<95 RON),"8,617","8,840",-223,-2.52%
Automotive gasoline total,"15,789","16,047",-258,-1.60%
Premium (95-97 RON),"1,980","2,010",-29,-1.45%
LPG Automotive use,239,243,-3,-1.32%
LPG total,"1,530","1,529",+1,+0.09%
LPG Non-automotive use,"1,291","1,286",+4,+0.35%
Diesel oil: total,"33,922","33,232",+691,+2.08%


## 6. Seasonality

Monthly index for each major product, using the last 5 complete years. Index 100 = annual mean. Helps you decide whether a recent monthly move reflects normal seasonal patterns or a genuine shift.

In [7]:
mp_full = demand[demand["product_native"].isin(MAJOR_PRODUCTS)].copy()
mp_full["month"] = mp_full["date"].dt.month
mp_full["year"] = mp_full["date"].dt.year

# Use only the last 5 COMPLETE calendar years so we measure CURRENT seasonality
# (post-COVID patterns), not 2010-vintage behaviour. `year < latest_year` drops
# the current incomplete year.
latest_year = mp_full["year"].max()
mp_recent = mp_full[mp_full["year"].between(latest_year - 5, latest_year - 1)]

monthly_mean = mp_recent.groupby(["product_native", "month"])["value"].mean()
annual_mean = mp_recent.groupby("product_native")["value"].mean()
monthly_idx = (monthly_mean / annual_mean * 100).reset_index(name="index_100")

fig = px.line(
    monthly_idx, x="month", y="index_100", color="product_native",
    title=f"Seasonality index (last 5 complete years {latest_year - 5}–{latest_year - 1};  100 = annual mean)",
    labels={"month": "Month", "index_100": "Seasonality index", "product_native": "Product"},
    markers=True,
)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.add_hline(y=100, line=dict(dash="dot", color="gray"))
fig.update_layout(template="plotly_white", height=460, hovermode="x unified")
fig.show()

## 6b. Seasonality — by calendar year

Same data, different lens. The chart in section 6 collapses every year into a single "average seasonal shape" — useful for asking _is this product's demand higher in summer?_ This chart keeps each year as its own line so you can answer the year-over-year shift question: _is 2026 tracking above, below, or in line with the historical seasonal pattern?_ The current year is highlighted.

This is the first chart in the notebook built from `analytics.charts.seasonality_by_year_chart` — the same function the India and (future) JODI dashboards will reuse.

In [8]:
# Add the project root to sys.path once. We use it for both `analytics` and
# `processors` imports in the cells below.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from analytics import seasonality_by_year_chart

# Take only the last ~7 calendar years so the chart isn't overloaded; the
# seasonal pattern hasn't shifted enough across 16 years for older years to
# add information. Tweak the cutoff if you want a longer historical view.
SEASONALITY_YEARS_BACK = 6
cutoff_year = demand["date"].max().year - SEASONALITY_YEARS_BACK

seas_df = demand[
    (demand["product_native"].isin(MAJOR_PRODUCTS))
    & (demand["date"].dt.year >= cutoff_year)
].copy()

fig = seasonality_by_year_chart(
    seas_df,
    products=MAJOR_PRODUCTS,
    value_col="value",
    date_col="date",
    product_col="product_native",
    cols=2,
    title=f"Australia demand — seasonality by calendar year ({cutoff_year}–{demand['date'].max().year}, latest in red)",
    units_label="ML",
)
fig.show()

## 7. Apparent vs reported (data-quality sanity check)

Reuses the same `compute_apparent_consumption` function the pipeline ships in `processors/australia_petroleum_statistics.py`. The formula is:

$$
\text{apparent} = \underbrace{\text{INDPROD} + \text{REFGROUT}}_{\text{domestic supply}} + \text{imports} - \text{exports} - \Delta\text{stocks}
$$

If apparent ≈ reported (within a few percent), the source data is internally consistent. A persistent large gap is a red flag for a unit mismatch, a missing flow, or an unreported revision. Note: a `: total` → bare-name product alias map inside the processor is a Phase 4 stopgap until proper canonicalisation lands.

In [9]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import processors.australia_petroleum_statistics as processor

ac = processor.compute_apparent_consumption(df)

# Look at the four products where this comparison is actually meaningful
# (Australia has both supply-side and demand-side rows after the Phase 4
# stopgap aliasing). For products without a supply-side counterpart,
# apparent_consumption will be 0 and the gap meaningless.
COMPARABLE_PRODUCTS = ["Diesel oil", "Automotive gasoline", "Aviation turbine fuel", "LPG"]

for product in COMPARABLE_PRODUCTS:
    sub = ac[ac["product_native"] == product].copy()
    if sub.empty:
        print(f"[skip] {product}: no rows")
        continue
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sub["date"], y=sub["apparent_consumption"],
        mode="lines", name="Apparent consumption (computed)",
    ))
    fig.add_trace(go.Scatter(
        x=sub["date"], y=sub["reported_sales"],
        mode="lines", name="Reported sales (TOTDEMO)", line=dict(dash="dash"),
    ))
    fig.update_layout(
        title=f"{product}: apparent vs reported (ML/month)",
        template="plotly_white", height=380, hovermode="x unified",
        yaxis_title="ML / month",
    )
    fig.show()

    # Average gap over the last 24 months — a single KPI per product
    recent_gap = sub.tail(24)["statdiff_pct"].dropna()
    if not recent_gap.empty:
        print(
            f"  {product:30s}  mean |statdiff_pct| last 24mo: "
            f"{recent_gap.abs().mean():.2f}%   "
            f"max: {recent_gap.abs().max():.2f}%"
        )

c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


  Diesel oil                      mean |statdiff_pct| last 24mo: 6.84%   max: 21.96%


  Automotive gasoline             mean |statdiff_pct| last 24mo: 8.81%   max: 26.38%


  Aviation turbine fuel           mean |statdiff_pct| last 24mo: 13.66%   max: 35.22%


  LPG                             mean |statdiff_pct| last 24mo: 39.70%   max: 105.60%


## 8. Interactive product deep-dive

Pick any product (all 42 demand-side labels are listed). Shows the full time series plus a 12-month rolling average. Pure render function — easy to port to Streamlit/Dash later by swapping the widget shell.

In [10]:
def render_product(product: str) -> None:
    """Plot a single product's demand time series + 12-month rolling mean.

    Pure function: no widget references, no globals beyond `demand`. That
    means a future Streamlit/Dash port is just a different UI shell over
    this same function.
    """
    sub = demand[demand["product_native"] == product].sort_values("date")
    if sub.empty:
        print(f"No data for {product!r}")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sub["date"], y=sub["value"], mode="lines", name=product,
    ))
    fig.add_trace(go.Scatter(
        x=sub["date"], y=sub["value"].rolling(12, min_periods=1).mean(),
        mode="lines", name="12-month rolling avg", line=dict(dash="dash"),
    ))
    fig.update_layout(
        title=f"{product} — monthly demand",
        template="plotly_white", height=420, hovermode="x unified",
        yaxis_title="ML / month",
    )
    fig.show()


product_picker = widgets.Dropdown(
    options=sorted(demand["product_native"].unique()),
    value="Diesel oil: total",
    description="Product:",
    layout=widgets.Layout(width="480px"),
)
out = widgets.Output()


def _on_change(change):
    if change["name"] == "value":
        with out:
            out.clear_output(wait=True)
            render_product(change["new"])


product_picker.observe(_on_change)
display(product_picker, out)
with out:
    render_product(product_picker.value)

Dropdown(description='Product:', index=6, layout=Layout(width='480px'), options=('Automotive gasoline total', …

Output()

## 9. DCCEEW vs JODI

JODI receives Australia's submission from DCCEEW each month, so in principle the two sources should agree. In practice there are small differences in timing, revision policy, and product coverage. This chart is the diagnostic.

**Alignment performed:**

- _Metric_: both filtered to `TOTDEMO` (reported sales/deliveries).
- _Unit_: both rendered in **kL** (kilolitres). We route JODI's `KBBL` column (unambiguously thousand-barrels) through `analytics.units.convert_series('kb', 'kL')` and multiply DCCEEW's ML by 1000. We deliberately avoid JODI's `KL` column because — confusingly — JODI's `unit_measure='KL'` values are reported in *megalitres* (thousands of the labelled unit), an SDMX-style convention easy to miss. Using `KBBL` removes the ambiguity.
- _Product_: matched on canonical *kind* (diesel, gasoline, jet, LPG, fuel oil) via `analytics.products.CANONICAL_AGGREGATE_LABELS`. DCCEEW's "Diesel oil: total" lines up with JODI's "GASDIES", etc.

A persistent gap on any panel is worth investigating (different product definitions? revision lag?).

In [11]:
from analytics import (
    cross_source_comparison_chart,
    convert_series,
    CANONICAL_AGGREGATE_LABELS,
    CANONICAL_KIND_LABEL,
)

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}.")
    print("        Run `python scripts/update_jodi.py` first.")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    # ------------------------------------------------------------------ #
    # Step 1: build matched DCCEEW + JODI frames keyed by canonical kind
    # ------------------------------------------------------------------ #
    # CANONICAL_AGGREGATE_LABELS tells us which native label per source
    # represents the kind-level total. For DCCEEW we MUST use "Diesel oil:
    # total" (not the bare "Diesel oil" supply-side label, and not
    # "Diesel oil: premium diesel" which is a sub-product).
    dceew_agg = CANONICAL_AGGREGATE_LABELS["dceew_petroleum_statistics"]
    jodi_agg = CANONICAL_AGGREGATE_LABELS["jodi"]
    # Only kinds that exist in BOTH sources can be compared.
    KINDS_TO_COMPARE = sorted(set(dceew_agg) & set(jodi_agg))
    print(f"Comparing {len(KINDS_TO_COMPARE)} kinds: {KINDS_TO_COMPARE}")

    # --- DCCEEW slice -------------------------------------------------- #
    dceew_panel = demand[demand["product_native"].isin(dceew_agg.values())].copy()
    # Attach the canonical kind label that the chart panels are titled by.
    # invert the dict so we can map native label -> kind in one step.
    dceew_kind_lookup = {v: k for k, v in dceew_agg.items()}
    dceew_panel["kind"] = dceew_panel["product_native"].map(dceew_kind_lookup)
    dceew_panel["panel"] = dceew_panel["kind"].map(CANONICAL_KIND_LABEL)
    # ML → kL is exact ×1000 (pure volume). Done inline because it's
    # simpler to read than calling convert_series for a single multiply.
    dceew_panel["value_kL"] = dceew_panel["value"] * 1000

    # --- JODI slice ---------------------------------------------------- #
    # NB: we pick JODI's KBBL column (thousand-barrels), NOT KL — JODI's
    # `unit_measure='KL'` values are actually in megalitres (an SDMX
    # convention), which would be off by ×1000 if used naively. KBBL is
    # unambiguous. We then route through analytics.units.convert_series
    # so the conversion factor lives in one place across the project.
    jodi_au = jodi[
        (jodi["ref_area"] == "AU")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBBL")
        & (jodi["energy_product"].isin(jodi_agg.values()))
    ].copy()
    jodi_kind_lookup = {v: k for k, v in jodi_agg.items()}
    jodi_au["kind"] = jodi_au["energy_product"].map(jodi_kind_lookup)
    jodi_au["panel"] = jodi_au["kind"].map(CANONICAL_KIND_LABEL)
    # kb → kL is pure volume (no density needed): 1 kb = 1000 bbl ×
    # 0.158987 m³/bbl = 158.987 m³ = 158.987 kL. The factor lives in
    # analytics.units.BBL_PER_M3.
    jodi_au["value_kL"] = convert_series(jodi_au["obs_value"], "kb", "kL")

    # The chart function needs the SAME panel labels in both frames so it
    # can match them; we already engineered that with the `panel` column.
    panel_order = [CANONICAL_KIND_LABEL[k] for k in KINDS_TO_COMPARE]

    fig = cross_source_comparison_chart(
        df_a=dceew_panel, df_b=jodi_au,
        products=panel_order,
        label_a="DCCEEW (Australia)", label_b="JODI",
        value_col_a="value_kL", value_col_b="value_kL",
        date_col_a="date", date_col_b="date",
        product_col_a="panel", product_col_b="panel",
        cols=2,
        title="Australia TOTDEMO — DCCEEW vs JODI (kL / month)",
        units_label="kL",
    )
    fig.show()

    # Mean absolute gap per kind, last 24 months — single-number diagnostic.
    print("\nMean absolute gap (DCCEEW vs JODI) over last 24 months, in kL:")
    for kind in KINDS_TO_COMPARE:
        panel = CANONICAL_KIND_LABEL[kind]
        d_kl = (
            dceew_panel.loc[dceew_panel["panel"] == panel, ["date", "value_kL"]]
            .set_index("date")["value_kL"]
        )
        j_kl = (
            jodi_au.loc[jodi_au["panel"] == panel, ["date", "value_kL"]]
            .set_index("date")["value_kL"]
        )
        merged = pd.concat(
            [d_kl.rename("dceew"), j_kl.rename("jodi")], axis=1, sort=False,
        ).dropna()
        recent = merged.tail(24)
        if recent.empty:
            print(f"  {panel:14s}  no overlap")
            continue
        gap = recent["dceew"] - recent["jodi"]
        pct = (gap.abs().mean() / recent["jodi"].abs().mean()) * 100
        print(
            f"  {panel:14s}  mean|gap| = {gap.abs().mean():>10,.0f} kL  "
            f"({pct:>5.2f}% of JODI level)"
        )

Comparing 4 kinds: ['diesel', 'fuel_oil', 'gasoline', 'lpg']



Mean absolute gap (DCCEEW vs JODI) over last 24 months, in kL:
  Diesel          mean|gap| =      7,540 kL  ( 0.27% of JODI level)
  Fuel oil        mean|gap| =      1,038 kL  ( 1.71% of JODI level)
  Gasoline        mean|gap| =     11,688 kL  ( 0.87% of JODI level)
  LPG             mean|gap| =      5,362 kL  ( 4.39% of JODI level)


## 10. Unit conversion demo

`analytics.units.convert` and `convert_series` are the single source of truth for unit conversion across all country dashboards. The cell below shows the same diesel demand line in five units so you can sanity-check the conversion factors and see the API in action.

**Supported units**

| Unit | Meaning | Type | Needs product_kind? | Needs date? |
| --- | --- | --- | --- | --- |
| `ML` | megalitres | volume | no | no |
| `kL` | kilolitres (= m³) | volume | no | no |
| `m3` | cubic metres | volume | no | no |
| `kb` | thousand barrels | volume | no | no |
| `kbd` | thousand barrels per day | rate | no | **yes** (days-in-month) |
| `kt` | kilotonnes | mass | **yes** (density) | no |

Conversion factors are IEA standard. See `analytics/units.py` for the table and `analytics/products.py` for the source-native-label → product-kind mapping.

In [12]:
from analytics import convert, convert_series, infer_product_kind

# Pick diesel as the demo product since it's the largest single demand
# category and exists in every source we'll plug in.
DEMO_LABEL = "Diesel oil: total"
DEMO_KIND = infer_product_kind(DEMO_LABEL, "dceew_petroleum_statistics")
print(f"{DEMO_LABEL!r}  →  product_kind = {DEMO_KIND!r}\n")

# Scalar examples — these are the API smoke tests you'd put in unit-test form.
print("Scalar examples for 1,000 ML over February 2026 (28 days):")
print(f"  1,000 ML → kL :  {convert(1000, 'ML', 'kL'):>14,.2f}")
print(f"  1,000 ML → kb :  {convert(1000, 'ML', 'kb'):>14,.4f}")
print(f"  1,000 ML → kbd:  {convert(1000, 'ML', 'kbd', date='2026-02-01'):>14,.4f}    ← per-day rate")
print(f"  1,000 ML → kt :  {convert(1000, 'ML', 'kt', product_kind=DEMO_KIND):>14,.4f}    ← needs density")

# Vectorised: take the full diesel demand series and stamp each row with
# the same product_kind so convert_series knows the density.
diesel = demand[demand["product_native"] == DEMO_LABEL].sort_values("date").copy()
diesel["kind"] = DEMO_KIND

# Three converted columns alongside the native ML column.
diesel["value_kL"] = diesel["value"] * 1000  # pure volume: × 1000 is exact
diesel["value_kb"] = convert_series(diesel["value"], "ML", "kb")
diesel["value_kbd"] = convert_series(diesel["value"], "ML", "kbd", date=diesel["date"])
diesel["value_kt"] = convert_series(
    diesel["value"], "ML", "kt",
    product_kind=diesel["kind"],  # per-row, in case different rows are different products
)

# Plot the same data in four units. Same SHAPE everywhere (because the
# data is the same!) — only the y-axis numbers change. That's the visual
# proof that the conversions are linear.
fig = make_subplots = None  # avoid name clash
import plotly.express as _px  # local alias keeps the snippet self-contained
import plotly.subplots as _sp
fig = _sp.make_subplots(
    rows=2, cols=2,
    subplot_titles=("ML / month", "kL / month", "kb / month (thousand barrels)", "kbd (thousand barrels / day)"),
    shared_xaxes=True,
)
for row, col, ycol, ylabel in [
    (1, 1, "value", "ML"),
    (1, 2, "value_kL", "kL"),
    (2, 1, "value_kb", "kb"),
    (2, 2, "value_kbd", "kbd"),
]:
    fig.add_trace(
        go.Scatter(x=diesel["date"], y=diesel[ycol], mode="lines", name=ylabel,
                   showlegend=False, line=dict(width=1.5)),
        row=row, col=col,
    )
    fig.update_yaxes(title_text=ylabel, row=row, col=col)
fig.update_layout(
    title=f"{DEMO_LABEL} — same data, four units",
    template="plotly_white", height=600,
)
fig.show()

# Quick numeric snapshot showing the conversion table for the most recent month.
latest = diesel.iloc[-1]
print(
    f"\nMost recent month ({latest['date'].strftime('%Y-%m')}):\n"
    f"  {latest['value']:>10,.1f}  ML\n"
    f"  {latest['value_kL']:>10,.1f}  kL\n"
    f"  {latest['value_kb']:>10,.1f}  kb\n"
    f"  {latest['value_kbd']:>10,.2f}  kbd  (over {latest['date'].days_in_month} days)\n"
    f"  {latest['value_kt']:>10,.1f}  kt  (using bbl/t={7.46} for diesel)"
)

'Diesel oil: total'  →  product_kind = 'diesel'

Scalar examples for 1,000 ML over February 2026 (28 days):
  1,000 ML → kL :    1,000,000.00
  1,000 ML → kb :      6,289.8100
  1,000 ML → kbd:        224.6361    ← per-day rate
  1,000 ML → kt :        843.1381    ← needs density



Most recent month (2026-03):
     3,014.6  ML
  3,014,600.0  kL
    18,961.3  kb
      611.65  kbd  (over 31 days)
     2,541.7  kt  (using bbl/t=7.46 for diesel)


## 11. DCCEEW vs Kayrros nowcaster — Australia jet fuel

A cross-source sanity check using the Kayrros flight-based nowcaster (`get_consumption` from `kayros/jet_fuel`). The nowcaster aggregates jet fuel **burned in flight** by aircraft departing Australia; DCCEEW reports jet fuel **sold / delivered** at Australian airports. In equilibrium these should be very close — fuel uplifted at an Australian airport is overwhelmingly burned on the flight that uplifted it — but they differ by tankering, in-airport stocks, DCCEEW revisions, and small coverage differences in the flight dataset.

**Alignment performed:**

- _Metric_: DCCEEW `Aviation turbine fuel total` (= Domestic + International sales) vs nowcaster total (all flights with `departure_country_name = 'Australia'`, passenger + cargo).
- _Unit_: both rendered in **kbd** (thousand barrels per day) via `analytics.units.convert_series`. kbd is the international oil-market standard and neutralises calendar-month length (so Feb doesn't read as a demand drop).
- _Coverage_: DCCEEW starts 2010-07; nowcaster starts 2018-12. The gap KPI and parity scatter use the overlap window only.

A persistent positive gap (nowcaster > DCCEEW) suggests tankering / stock draws / nowcaster over-coverage; a persistent negative gap suggests stock builds or nowcaster under-coverage. A widening fan over time is the signal to investigate.

In [13]:
import sys, os, duckdb

# Path to the jet_fuel project root (the folder that contains src/, data/, etc.)
sys.path.insert(0, r"C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\kayros\jet_fuel")

# Path to the DuckDB file (optional, but recommended — see note below)
os.environ['JET_FUEL_DB_PATH'] = r"C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\kayros\jet_fuel\data\jet_fuel.duckdb"

from src.export import get_consumption

# India monthly totals for PPAC comparison
india = get_consumption(scope_type='country', scope='India', freq='monthly')

# Multi-country wide-format kbd
df = get_consumption(
    scope_type='country',
    scope=['Australia'],
    freq='monthly',
    metric='total_kb',
    pivot=True,
    drop_incomplete=True,   # skip in-progress trailing month
)

In [14]:
df#[df.columns] = df[df.columns].div(7.9)

,period_start,Australia
0,2018-12-01,4951.931166
1,2019-01-01,5072.747943
2,2019-02-01,4417.135537
3,2019-03-01,4744.051888
4,2019-04-01,4602.973886
...,...,...
84,2025-12-01,5408.192162
85,2026-01-01,5423.164814
86,2026-02-01,4781.666323
87,2026-03-01,4593.623555


In [15]:
"""Section 11: DCCEEW (official) vs Kayrros nowcaster — Australia jet fuel.

Both sources are rendered in kbd (thousand barrels per day). Assumes prior
cells have already run:
  - `demand` — DCCEEW long-format frame (Section 1).
  - `df`     — wide-format nowcaster output for Australia in kb (kayros
               import cell above).
"""
from analytics import convert_series, cross_source_gap_chart

# DCCEEW jet, monthly. 'Aviation turbine fuel total' = Domestic + International,
# which is the right counterpart for the nowcaster: the nowcaster sees every
# flight departing Australia, regardless of destination.
dcceew_jet = (
    demand[demand["product_native"] == "Aviation turbine fuel total"]
    .loc[:, ["date", "value"]]
    .sort_values("date")
    .reset_index(drop=True)
    .rename(columns={"value": "ML"})
)
# Direct ML→kbd via analytics.units; the helper does the days-in-month divide
# internally so we don't reimplement it (and don't drift from JODI comparison).
dcceew_jet["kbd"] = convert_series(
    dcceew_jet["ML"], "ML", "kbd", date=dcceew_jet["date"],
)

# Nowcaster is already in kb (metric='total_kb'). Same helper routes kb→kbd.
nowcaster = (
    df.rename(columns={"period_start": "date", "Australia": "kb"})
    .loc[:, ["date", "kb"]]
    .sort_values("date")
    .reset_index(drop=True)
)
nowcaster["kbd"] = convert_series(
    nowcaster["kb"], "kb", "kbd", date=nowcaster["date"],
)

# Inner-merge on month gives the overlap window for the gap KPI and parity
# scatter. Each source keeps its own full line on the time-series chart so
# the viewer can still see where each begins/ends.
overlap = (
    dcceew_jet.merge(
        nowcaster, on="date", how="inner", suffixes=("_dcceew", "_now"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

# ── Levels + gap chart (shared analytics.charts helper) ────────────────────
# Direction b_minus_a = nowcaster − DCCEEW: positive means flights burned more
# fuel than DCCEEW recorded as sold (tankering / stock draws). The [Absolute]
# / [Percent] toggle in the top-left of the chart flips the gap panel between
# kbd and % of DCCEEW without re-rendering; hover always shows both numbers.
fig = cross_source_gap_chart(
    dcceew_jet, nowcaster,
    label_a="DCCEEW",
    label_b="Kayrros",
    value_col_a="kbd", value_col_b="kbd",
    gap_direction="b_minus_a",
    units_label="kbd",
    title="Australia jet fuel: DCCEEW vs Kayrros nowcaster",
    height=620,
)
fig.show()

# ── Gap KPI ────────────────────────────────────────────────────────────────
# gap = nowcaster − DCCEEW. Positive means the nowcaster sees more fuel burn
# than DCCEEW records as sold. Last 24 overlapping months keeps the focus on
# the recent regime (post-COVID recovery), where the comparison is most
# operationally useful.
overlap["gap_kbd"] = overlap["kbd_now"] - overlap["kbd_dcceew"]
overlap["gap_pct"] = overlap["gap_kbd"] / overlap["kbd_dcceew"] * 100

recent = overlap.tail(24).dropna(subset=["gap_pct"])
print(
    f"Overlap window: {overlap['date'].min().date()} → "
    f"{overlap['date'].max().date()}  ({len(overlap)} months)"
)
print()
print(f"Last {len(recent)} overlapping months — nowcaster vs DCCEEW:")
print(
    f"  mean signed gap : {recent['gap_kbd'].mean():+7.2f} kbd  "
    f"({recent['gap_pct'].mean():+6.2f}% of DCCEEW)"
)
print(
    f"  mean |gap|      : {recent['gap_kbd'].abs().mean():7.2f} kbd  "
    f"({recent['gap_pct'].abs().mean():6.2f}% of DCCEEW)"
)
print(
    f"  max  |gap|      : {recent['gap_kbd'].abs().max():7.2f} kbd  "
    f"({recent['gap_pct'].abs().max():6.2f}% of DCCEEW)"
)

# ── Parity scatter (nowcaster vs DCCEEW) ───────────────────────────────────
# 45° line = perfect agreement. Markers coloured chronologically so a recent
# drift (e.g. nowcaster coverage shift, COVID structural break) is visible as
# a colour-graded fan moving away from the line, not just as a uniform cloud.
lo = float(min(overlap["kbd_dcceew"].min(), overlap["kbd_now"].min())) * 0.95
hi = float(max(overlap["kbd_dcceew"].max(), overlap["kbd_now"].max())) * 1.05

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=overlap["kbd_dcceew"], y=overlap["kbd_now"],
    mode="markers",
    marker=dict(
        size=8,
        color=overlap.index,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="month index<br>(early→late)"),
    ),
    customdata=overlap["date"].dt.strftime("%Y-%m"),
    hovertemplate=(
        "%{customdata}<br>DCCEEW: %{x:,.2f} kbd"
        "<br>Nowcaster: %{y:,.2f} kbd<extra></extra>"
    ),
    name="Months",
))
fig2.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi], mode="lines",
    line=dict(color="grey", dash="dash"),
    name="45° (perfect agreement)", hoverinfo="skip",
))
fig2.update_layout(
    title="Parity: nowcaster vs DCCEEW (Australia jet, kbd)",
    template="plotly_white", height=520,
    xaxis=dict(title="DCCEEW kbd", range=[lo, hi]),
    yaxis=dict(
        title="Nowcaster kbd", range=[lo, hi],
        scaleanchor="x", scaleratio=1,
    ),
)
fig2.show()

Overlap window: 2018-12-01 → 2026-03-01  (88 months)

Last 24 overlapping months — nowcaster vs DCCEEW:
  mean signed gap :  -12.66 kbd  ( -7.45% of DCCEEW)
  mean |gap|      :   12.66 kbd  (  7.45% of DCCEEW)
  max  |gap|      :   20.97 kbd  ( 11.64% of DCCEEW)


---

## Where to take this next

- **State-level demand**: the source xlsx has state-by-state breakdowns we haven't parsed yet — extend `scrapers/australia_apstat.py::SHEETS_CONFIG` to pick up `Sales by state` sheets, then add a state filter to the dashboard.
- **Decomposition**: a proper STL decomposition (`statsmodels.tsa.seasonal.STL`) gives trend + seasonal + residual cleanly, vs. the rolling-mean approximation here.
- **Cross-country**: once Phase 4 lands and India + JODI emit the canonical schema, the same render functions will work with `country` as just another dimension.
- **Forecast**: a Prophet or SARIMAX on top of `demand_products` gets you a 12-month-out point + uncertainty band per product.